# 09 — Spark ML: Tuning and Productionizing

Hyperparameter search with `ParamGridBuilder` + `CrossValidator`, saving/loading fitted pipelines, and the "when would you actually reach for Spark ML vs. scikit-learn" + "how would you serve this model" questions that close out most Spark ML interviews.

> **Setup note:** these notebooks are written but **not executed** — PySpark is not
> installed in this environment. To run them locally:
>
> ```bash
> python -m venv .venv && source .venv/bin/activate
> pip install pyspark==3.5.1
> # Java 11/17 must be on PATH (java -version)
> jupyter notebook
> ```
>
> Everything below is correct, runnable PySpark MLlib (`pyspark.ml`) — read it as
> a reference and run cell-by-cell once your environment is set up.

In [ ]:
from pyspark.sql import SparkSession
from pyspark.ml import Pipeline
from pyspark.ml.feature import StringIndexer, OneHotEncoder, VectorAssembler
from pyspark.ml.classification import LogisticRegression

spark = (
    SparkSession.builder.appName("spark-ml-tuning").master("local[*]")
    .config("spark.sql.shuffle.partitions", "8").getOrCreate()
)

customers = spark.createDataFrame(
    [
        (1, "month-to-month", 29.99, 2, 0), (2, "one-year", 19.50, 24, 0),
        (3, "month-to-month", 89.10, 1, 1), (4, "two-year", 15.00, 36, 0),
        (5, "month-to-month", 95.40, 3, 1), (6, "one-year", 10.00, 18, 0),
        (7, "month-to-month", 75.00, 4, 1), (8, "two-year", 22.00, 30, 0),
        (9, "month-to-month", 60.00, 2, 1), (10, "one-year", 18.00, 20, 0),
    ],
    ["customer_id", "contract_type", "monthly_charges", "tenure_months", "churned"],
)
train, test = customers.randomSplit([0.8, 0.2], seed=7)

pipeline = Pipeline(stages=[
    StringIndexer(inputCol="contract_type", outputCol="contract_idx"),
    OneHotEncoder(inputCols=["contract_idx"], outputCols=["contract_vec"]),
    VectorAssembler(inputCols=["contract_vec", "monthly_charges", "tenure_months"], outputCol="features"),
    LogisticRegression(featuresCol="features", labelCol="churned"),
])

## 1. `ParamGridBuilder` + `CrossValidator`

`ParamGridBuilder` defines the hyperparameter search space over any stage in the pipeline (reference the stage's param directly, e.g. `logreg.regParam`). `CrossValidator` then does **k-fold** cross-validation: splits `train` into `k` folds, and for **every** combination in the grid, trains on `k-1` folds and validates on the held-out fold, `k` times, then averages the evaluator's metric. It picks the best combination and refits a final model on the **entire** training set with those params.

**Cost warning (a real interview question):** total fits = `len(grid) * numFolds`. A 3x3 grid with `numFolds=3` is **27** full pipeline fits — expensive when the pipeline includes a full distributed training job. `TrainValidationSplit` does a single train/validation split instead of k folds — `len(grid)` fits total, much cheaper, at the cost of a noisier estimate of each configuration's quality (relevant to mention when data is large enough that a single split is already a reliable estimate).

In [ ]:
from pyspark.ml.tuning import ParamGridBuilder, CrossValidator
from pyspark.ml.evaluation import BinaryClassificationEvaluator
from pyspark.ml.classification import LogisticRegression

logreg_stage = pipeline.getStages()[-1]

param_grid = (
    ParamGridBuilder()
    .addGrid(logreg_stage.regParam, [0.0, 0.1, 0.5])
    .addGrid(logreg_stage.elasticNetParam, [0.0, 1.0])
    .build()
)
# grid size here: 3 * 2 = 6 combinations

cv = CrossValidator(
    estimator=pipeline,
    estimatorParamMaps=param_grid,
    evaluator=BinaryClassificationEvaluator(labelCol="churned"),
    numFolds=3,       # 6 combos * 3 folds = 18 full pipeline fits
    parallelism=2,    # fit multiple param combos concurrently
)

cv_model = cv.fit(train)
best_pipeline_model = cv_model.bestModel
print("avg metric per combo:", cv_model.avgMetrics)

In [ ]:
from pyspark.ml.tuning import TrainValidationSplit

# Cheaper alternative: single 80/20 train/validation split instead of k folds
tvs = TrainValidationSplit(
    estimator=pipeline,
    estimatorParamMaps=param_grid,
    evaluator=BinaryClassificationEvaluator(labelCol="churned"),
    trainRatio=0.8,   # 6 combos * 1 split = 6 fits, vs 18 for 3-fold CV above
)
tvs_model = tvs.fit(train)

## 2. Persisting models

A production DE pipeline typically **trains offline** (a batch/nightly Spark job) and **scores elsewhere** — so saving/loading the fitted `PipelineModel` is a core skill, not an afterthought. Every stage's learned parameters (indexer mappings, scaler stats, model coefficients) are serialized together, so `.transform()` on new data reproduces the exact same feature engineering + scoring used at training time — critical for training/serving consistency.

In [ ]:
model_path = "/tmp/churn_pipeline_model"

# best_pipeline_model.write().overwrite().save(model_path)

from pyspark.ml import PipelineModel
# loaded_model = PipelineModel.load(model_path)
# loaded_model.transform(new_customers_df).select("customer_id", "prediction").show()

## 3. Spark ML vs. scikit-learn — when to actually use which

A very common interview question, since most ML practitioners default to sklearn:

| | Spark ML | scikit-learn |
|---|---|---|
| Scale | trains directly on data that doesn't fit on one machine | needs data to fit in one machine's RAM |
| Algorithm coverage | narrower — linear models, trees/forests/GBTs, k-means, ALS, basic NLP | much broader (SVMs, gradient boosting variants, huge ecosystem) |
| Ecosystem/tuning tools | fewer (basic grid/random search via `ParamGridBuilder`) | huge (Optuna, extensive diagnostics, `imbalanced-learn`, etc.) |
| Where it shines | **feature engineering at scale** even if the final model is small; training when data is genuinely too big for one node |

**Common real pattern:** use Spark for the heavy distributed ETL/feature engineering, then downsample/aggregate to something that fits in memory and train with sklearn/XGBoost for the final model — you get Spark's scale where it matters and sklearn's richer tooling where it matters. Reasonable answer if asked "would you always use Spark ML once you're already in Spark?": **no** — only when training itself needs to be distributed.

## 4. Serving — the limitation interviewers probe for

Spark ML models are built for **batch/offline scoring**: call `.transform()` on a DataFrame of new records inside a Spark job. They are a poor fit for **low-latency, single-record, real-time serving** — every `.transform()` call goes through JVM/Spark session overhead unsuitable for a sub-100ms API request path.

For real-time serving, the typical pattern is exporting the trained model to a lightweight, dependency-free format and serving it outside Spark entirely — e.g. **MLeap** or **ONNX** export, or, more simply, extracting just the learned coefficients/tree structure and reimplementing scoring in a small service. Experiment tracking across either path (Spark ML training or a downstream sklearn model) is commonly done with **MLflow** — see `../../mlops_aiops/docs/tools/mlflow/` in this repo for a deeper look at that piece.

## 5. Interview Q&A

1. **"Why is `CrossValidator` expensive and how would you reduce the cost?"** — total fits = grid size × folds; reduce by shrinking the grid, using `TrainValidationSplit` instead of k-fold, or raising `parallelism` to fit combinations concurrently (bounded by cluster resources).
2. **"How do you guarantee training/serving consistency?"** — persist and reload the entire fitted `PipelineModel` (feature engineering + model together), never reimplement feature logic separately for serving.
3. **"Would you serve real-time predictions directly from a Spark ML model?"** — no; Spark ML is built for batch scoring. For low-latency serving, export to a lightweight format (MLeap/ONNX) or reimplement scoring outside Spark.
4. **"Your CrossValidator run OOMs / is too slow on a huge dataset — what would you check first?"** — the same performance levers as any Spark job (notebook 05): partition count, shuffle/spill in the training stages, and whether feature engineering stages are being unnecessarily recomputed rather than cached across folds.

## Summary

- `CrossValidator` = k-fold, most thorough, most expensive; `TrainValidationSplit` = single split, cheaper, noisier estimate.
- Save/load the whole `PipelineModel`, not just the final model stage — that's what keeps training and serving feature logic identical.
- Reach for Spark ML when training itself must be distributed; otherwise sklearn's ecosystem is usually the better tool downstream of Spark-based feature engineering.
- Spark ML serves batch predictions well; real-time serving needs an export step (MLeap/ONNX) or a separate lightweight scoring service.
- This closes the Spark ML sequence — see `01`-`06` for Spark core/SQL, and `../../sql_postgres_practice/` for the SQL side of the same interview prep.